# 01 — Grafo ponderado: el motor que decide antes del LLM

**Hipótesis** (init.md): con las decisiones editoriales tomadas por el grafo (narrativa, tono,
audiencia, temas, amenidades prioritarias), el prompt baja de ~700 a ~520 tokens y el modelo
solo redacta — mejor obediencia, menos invención, y la puerta abierta a bajar de tamaño de
modelo sin perder calidad.

**Piezas:**
- `motor_inferencia.py` — el motor oficial (mismo comportamiento que el simulador del
  visualizador): activación → propagación ponderada → bloqueos → selección top-k → resumen.
- `test_motor.py` — 67 tests (decisión, bloqueos exhaustivos, carga). Correr con
  `pytest test_motor.py` antes de tocar pesos.
- `datasets/` — la única fuente de conocimiento (catálogo real de la BD).

**Variante de este experimento:** v4-4B contra los baselines v3 de `notebooks/llm/`.
(v4-1.7B queda como experimento siguiente si v4-4B valida el enfoque: cambiar la ruta del
GGUF y añadir `/no_think` al system prompt.)

## 1. Carga del dominio y decisiones de los 7 drafts

El motor es determinista y corre en microsegundos — todo esto es pre-LLM.

In [1]:
import json
import re
import time
from pathlib import Path

import pandas as pd
from motor_inferencia import (
    cargar_dominio, cargar_drafts, inferir, resumen_para_llm, draft_json_a_texto,
)

DOM = cargar_dominio("datasets")
DRAFTS = cargar_drafts("datasets/drafts_ejemplo.json")
print(f"Dominio: {len(DOM.amenidades)} amenidades, {len(DOM.temas)} temas, "
      f"{len(DOM.tipos)} tipos, {len(DOM.audiencias)} audiencias, {len(DOM.aristas)} aristas")

t0 = time.perf_counter()
decisiones = {d["id"]: inferir(DOM, d) for d in DRAFTS}
print(f"7 inferencias del grafo en {(time.perf_counter() - t0) * 1000:.2f} ms\n")

pd.DataFrame([{
    "draft": dec.draft_id,
    "temas": ", ".join(t.replace("_", " ") for t in dec.temas),
    "audiencia": dec.audiencia,
    "protagonistas": ", ".join(dec.protagonistas),
    "bloqueados": ", ".join(dec.bloqueados),
    "desconocidas": ", ".join(dec.desconocidas),
} for dec in decisiones.values()])

Dominio: 22 amenidades, 11 temas, 6 tipos, 7 audiencias, 80 aristas
7 inferencias del grafo en 0.65 ms



,draft,temas,audiencia,protagonistas,bloqueados,desconocidas
0,draft-001,"vida al aire libre, convivencia familiar",familia_grande,"Terraza, Jardín",patrimonio,
1,draft-002,"seguridad, practicidad",familia_pareja,"Seguridad 24h, Elevador",flexibilidad,
2,draft-003,"practicidad, convivencia familiar",familia_grande,"Cocina Integral, Cuarto de Servicio","flexibilidad, estilo_moderno",
3,draft-004,"flexibilidad, practicidad",profesionista,"Amueblado, Closets","patrimonio, convivencia_familiar",
4,draft-005,"convivencia familiar, tranquilidad",familia_pareja,,patrimonio,
5,draft-006,"vida al aire libre, convivencia familiar",familia_pareja,"Terraza, Área de Juegos Infantiles",flexibilidad,Sala de cine
6,draft-007,"seguridad, funcionalidad comercial",negocio_emprendedor,"Circuito Cerrado (CCTV), Acceso Controlado","patrimonio, convivencia_familiar, tranquilidad",


In [2]:
# La traza: por qué el grafo decidió lo que decidió (auditabilidad para negocio).
print("═══ TRAZA draft-007 (Local Comercial) ═══")
for linea in decisiones["draft-007"].traza:
    print(" ", linea)

print("\n═══ RESUMEN PARA EL LLM — draft-001 ═══")
print(resumen_para_llm(DOM, DRAFTS[0], decisiones["draft-001"]))
print("\n═══ RESUMEN PARA EL LLM — draft-007 ═══")
print(resumen_para_llm(DOM, DRAFTS[6], decisiones["draft-007"]))

═══ TRAZA draft-007 (Local Comercial) ═══
  activaciones: RENTA, Local Comercial, construccion_reciente, sin_estacionamiento, cctv, acceso_controlado, aire_acondicionado
  RENTA → flexibilidad (+0.6)
  RENTA → aud:profesionista (+0.3)
  BLOQUEO patrimonio ⊣ RENTA: hablar de patrimonio e inversión de largo plazo es discurso de venta
  Local Comercial → funcionalidad_comercial (+0.8)
  Local Comercial → aud:negocio_emprendedor (+0.9)
  BLOQUEO convivencia_familiar ⊣ Local Comercial: un inmueble comercial no se narra con vida familiar
  BLOQUEO tranquilidad ⊣ Local Comercial: a un local le conviene flujo y actividad; no se vende con quietud
  cctv → seguridad (+0.9)
  cctv → funcionalidad_comercial (+0.3)
  acceso_controlado → tranquilidad (+0.3)
  acceso_controlado → seguridad (+0.9)
  aire_acondicionado → confort_interior (+0.9)
  aire_acondicionado → funcionalidad_comercial (+0.3)
  temas ganadores (k=2, umbral=0.5): ['seguridad', 'funcionalidad_comercial']
  audiencia: negocio_emprend

## 2. Prompt v4: obedecer, no elegir

El system prompt se recorta: ya no instruye al modelo a *elegir* ángulos (eso lo hizo el
grafo), solo a *obedecer* las DECISIONES DE REDACCIÓN. Reglas duras intactas.

In [3]:
from llama_cpp import Llama

SYSTEM_PROMPT_V4 = """Eres un redactor inmobiliario profesional de México. Recibirás el DRAFT de una \
propiedad y un bloque de DECISIONES DE REDACCIÓN ya tomadas. Tu único trabajo es redactar el \
anuncio obedeciéndolas.

Reglas estrictas:
1. Escribe SOLO con la información del DRAFT y de las DECISIONES. No agregues características, \
lugares, vistas ni cualidades que no aparezcan ahí.
2. Desarrolla los temas en el orden indicado parafraseando sus frases con naturalidad. Destaca \
las amenidades protagonistas; las secundarias solo de pasada.
3. PROHIBIDO: precios o cifras monetarias; datos de contacto; invitaciones a llamar, escribir o \
visitar; lenguaje de urgencia ("aprovecha", "no te lo pierdas", "oportunidad única").
4. Ubicación: solo el nombre de la colonia. Di "recámaras" y "estacionamientos" (vocabulario \
de la región). Si el draft no menciona estacionamientos, no hables de ellos.
5. Título de máximo 10 palabras. Descripción de 21 a 70 palabras, párrafos fluidos, tono cálido \
y humano, sin mayúsculas sostenidas ni signos de admiración excesivos.

Responde exclusivamente con un JSON con las claves "titulo" y "descripcion"."""

ESQUEMA_ANUNCIO = {
    "type": "object",
    "properties": {"titulo": {"type": "string"}, "descripcion": {"type": "string"}},
    "required": ["titulo", "descripcion"],
}


def mensaje_v4(draft: dict) -> str:
    decision = inferir(DOM, draft)
    return (f"DRAFT DE LA PROPIEDAD:\n{draft_json_a_texto(draft)}\n\n"
            f"{resumen_para_llm(DOM, draft, decision)}")


# Medición de tokens con solo el tokenizador del 4B (sin cargar los pesos del modelo)
tok = Llama(model_path="../../models_registry/llm/Qwen3-4B-Instruct-2507-Q4_K_M.gguf",
            vocab_only=True, verbose=False)
n_sys = len(tok.tokenize(SYSTEM_PROMPT_V4.encode()))
n_msgs = [len(tok.tokenize(mensaje_v4(d).encode())) for d in DRAFTS]
print(f"system prompt v4: {n_sys} tokens (v3 tenía ~340)")
print(f"mensaje draft+decisiones: prom {sum(n_msgs) / len(n_msgs):.0f} tokens (v3 tenía ~330)")
print(f"prompt total estimado: ~{n_sys + sum(n_msgs) / len(n_msgs):.0f} tokens (objetivo ≤ ~550; v3 usaba ~700)")
del tok

llama_context: n_ctx_seq (512) > n_ctx_train (0) -- possible training context overflow


system prompt v4: 316 tokens (v3 tenía ~340)
mensaje draft+decisiones: prom 261 tokens (v3 tenía ~330)
prompt total estimado: ~577 tokens (objetivo ≤ ~550; v3 usaba ~700)


## 3. Corrida v4 sobre los 7 drafts

Elige la variante en la variable `VARIANTE` de la celda (`v4-4B` o `v4-1.7B`, esta última con
`/no_think` automático). Se monta **un solo modelo a la vez** y los resultados se acumulan en
`resultados_v4.csv` sin pisar los de otras variantes. Cierra los kernels de otros notebooks
antes de correr (el 4B ocupa ~5 GB; el 1.7B ~2.3 GB).

In [4]:
# Configura aquí la variante a correr; los resultados se ACUMULAN en resultados_v4.csv
# (solo se reemplazan las filas de la misma variante).
VARIANTE = "v4-1.7B"
MODELO = {
    "v4-4B": ("../../models_registry/llm/Qwen3-4B-Instruct-2507-Q4_K_M.gguf", ""),
    "v4-1.7B": ("../../models_registry/llm/Qwen3-1.7B-Q4_K_M.gguf", " /no_think"),
}
RUTA_GGUF, SUFIJO_SISTEMA = MODELO[VARIANTE]

RE_PRECIO = re.compile(r"\$|\bprecio\b|\bmxn\b|\bpesos?\b|mensualidad|mill[oó]n|\bmonto\b", re.I)
RE_CONTACTO = re.compile(
    r"cont[aá]ct|ll[aá]m[ae]|tel[eé]fono|whatsapp|escr[ií]b[ae]|agend[ae]|vis[ií]t[ae]|"
    r"aprovecha|no te lo pierdas|[uú]ltimos d[ií]as|cita|oportunidad [uú]nica", re.I)

llm = Llama(model_path=RUTA_GGUF, n_ctx=4096, n_threads=4, verbose=False)

filas = []
for draft in DRAFTS:
    t0 = time.perf_counter()
    salida = llm.create_chat_completion(
        messages=[{"role": "system", "content": SYSTEM_PROMPT_V4 + SUFIJO_SISTEMA},
                  {"role": "user", "content": mensaje_v4(draft)}],
        response_format={"type": "json_object", "schema": ESQUEMA_ANUNCIO},
        temperature=1.0,
        max_tokens=512,
    )
    dt = round(time.perf_counter() - t0, 1)
    anuncio = json.loads(salida["choices"][0]["message"]["content"])
    texto = f"{anuncio['titulo']} {anuncio['descripcion']}"
    fila = {"draft_id": draft["id"], "variante": VARIANTE,
            "titulo": anuncio["titulo"], "descripcion": anuncio["descripcion"],
            "tokens_prompt": salida["usage"]["prompt_tokens"], "tiempo_s": dt,
            "n_palabras": len(anuncio["descripcion"].split()),
            "menciona_precio": bool(RE_PRECIO.search(texto)),
            "menciona_contacto": bool(RE_CONTACTO.search(texto))}
    filas.append(fila)
    marca = "⚠️" if fila["menciona_precio"] or fila["menciona_contacto"] else "✓"
    print(f"{marca} [{VARIANTE}] {draft['id']}: {anuncio['titulo']} "
          f"({dt} s, prompt {fila['tokens_prompt']} tok)")

df_corrida = pd.DataFrame(filas)
if Path("resultados_v4.csv").exists():
    previas = pd.read_csv("resultados_v4.csv")
    df_v4 = pd.concat([previas[previas["variante"] != VARIANTE], df_corrida], ignore_index=True)
else:
    df_v4 = df_corrida
df_v4.to_csv("resultados_v4.csv", index=False)
print(f"\nGuardado en resultados_v4.csv ({df_v4.groupby('variante').size().to_dict()})")

✓ [v4-1.7B] draft-001: Casa en Prudencio Moscoso (20.3 s, prompt 596 tok)
✓ [v4-1.7B] draft-002: Departamento en El Cerrillo (12.4 s, prompt 598 tok)
✓ [v4-1.7B] draft-003: Casa de 320 m² en Barrio de Guadalupe (15.2 s, prompt 595 tok)
✓ [v4-1.7B] draft-004: Departamento Práctico (8.9 s, prompt 614 tok)
✓ [v4-1.7B] draft-005: Casa 31 de Marzo (9.7 s, prompt 538 tok)
✓ [v4-1.7B] draft-006: Dpto. Con Terraza y Juegos Infantiles (12.2 s, prompt 619 tok)
✓ [v4-1.7B] draft-007: Espacio Comercial Seguro y Funcional (14.1 s, prompt 596 tok)

Guardado en resultados_v4.csv ({'v4-1.7B': 7, 'v4-4B': 7})


## 4. Comparativa contra los baselines v3 y lectura lado a lado

In [ ]:
marcos = [df_v4]
if Path("../llm/resultados_ab_grafo.csv").exists():
    ab = pd.read_csv("../llm/resultados_ab_grafo.csv")
    marcos.append(ab[ab["variante"] == "v3"].assign(variante="v3-4B"))
if Path("../llm/resultados_qwen17b.csv").exists():
    marcos.append(pd.read_csv("../llm/resultados_qwen17b.csv"))
df_comp = pd.concat(marcos, ignore_index=True)

print("--- Resumen por variante ---")
display(df_comp.groupby("variante")[["menciona_precio", "menciona_contacto",
                                      "tokens_prompt", "tiempo_s", "n_palabras"]].agg(
    {"menciona_precio": "sum", "menciona_contacto": "sum", "tokens_prompt": "mean",
     "tiempo_s": "mean", "n_palabras": "mean"}).round(1))

for draft_id in ("draft-001", "draft-007"):
    print(f"\n{'#' * 70}\n# {draft_id}\n{'#' * 70}")
    for variante in ("v3-4B", "v3-1.7B", "v4-4B", "v4-1.7B"):
        fila = df_comp[(df_comp["draft_id"] == draft_id) & (df_comp["variante"] == variante)]
        if fila.empty:
            continue
        fila = fila.iloc[0]
        print(f"\n### {variante} — {fila['titulo']} ({fila['tiempo_s']} s)\n{fila['descripcion']}")

--- Resumen por variante ---


,menciona_precio,menciona_contacto,tokens_prompt,tiempo_s,n_palabras
variante,,,,,
v3-1.7B,0,0,673.6,12.9,63.9
v3-4B,0,0,669.6,26.1,59.7
v4-1.7B,0,0,593.7,13.3,66.7
v4-4B,0,0,614.3,25.4,59.3



######################################################################
# draft-001
######################################################################

### v3-4B — Casa de reciente construcción en Prudencio Moscoso (35.7 s)
Una casa amplia y moderna, a estrenar, en el corazón de Prudencio Moscoso. Ofrece espacio para vivir con tranquilidad y bienestar. Cuenta con terraza para disfrutar el aire libre sin salir de casa y un gimnasio para mantener una rutina activa y saludable. Ideal para familias que buscan un hogar con independencia y espacios para todos.

### v3-1.7B — Casa de 200 m² en Prudencio Moscoso (12.6 s)
La casa de 200 m² en Prudencio Moscoso ofrece espacio amplio para vivir en un ambiente acogedor. Con 4 recámaras, 3 baños y 2 estacionamientos, es ideal para familias que buscan independencia y comodidad. La terraza y el gimnasio permiten disfrutar de la vida al aire libre y mantener una rutina activa. A estrenar, su diseño moderno combina funcionalidad y estilo, garantiza

: 

## 5. Evaluación (criterios del init.md §7)

| Métrica | Objetivo v4-4B | Resultado |
|---|---|---|
| Tokens de prompt | ≤ ~550 (v3: ~700) | |
| Tiempo por anuncio | < 26 s de v3-4B (menos prompt que procesar) | |
| Violaciones precio/contacto | 0/7 | |
| Tests del motor (`pytest test_motor.py`) | 100% verdes | ✅ 67/67 |
| Calidad vs v3-4B (lectura lado a lado) | igual o mejor, con más obediencia | |

- [ ] **Obediencia:** ¿el anuncio desarrolla los temas en el orden decidido y destaca las
      protagonistas? (comparar contra la columna de decisiones de la sección 1)
- [ ] **Fidelidad:** ¿nada fuera del draft + decisiones?
- [ ] **Comercial:** ¿draft-007 suena a local (operación, clientes) y no a hogar?

**Si v4-4B valida el enfoque:** repetir esta corrida con el 1.7B (`/no_think`) — la hipótesis
de fondo es que con las decisiones ya tomadas, el modelo chico alcanza esta misma calidad a
~11 s/anuncio. Con el ganador: benchmark en el VPS y el worker de `llm_queue` consumiendo
`motor_inferencia.py` + `datasets/` como artefactos versionados.

**Si los pesos necesitan ajuste:** editar `datasets/*.csv` → `pytest test_motor.py` →
regenerar visualizador (`python generar_visualizador.py`) → repetir sección 3.